# ANAEMIA ANALYSIS USING NATIONAL HEALTH AND NUTRITION EXAMINATION SURVEY (NHANES) DATA.


## 1.0 Introduction

### 1.1 Research Question
Among NHANES participants with anaemia, which observable multidimensional biological patterns are consistent with plausible aetiological mechanisms of multifactorial anaemia, and to what extent do these patterns overlap? 

### 1.2 Research Objective
Use routinely collected multidimensional biological data to identify patterns that may distinguish observable, mechanism-related phenotypes of multifactorial anaemia and examine how often or the extent to which these patterns overlap.

#### 1.2.1 Specific Objectives
The goal of this analysis is to:
1. Characterise participants with phenotypes and biological measurements consistent with varying plausible aetiological mechanisms of complex anaemia.
2. Identify distinct biological patterns among NHANES participants with anaemia using exploratory and unsupervised learning methods.
3. Investigate the extent to which these patterns overlap with one another and align with scientifically established thresholds and biological mechanisms associated with anaemia arising from multiple underlying causes.

### 1.3 Dataset Overview
- Source: The US National Health and Nutrition Examination Survey, conducted by the National Center for Health Statistics (NCHS), a component of the Centers for Disease Control and Prevention (CDC).
- Survey period: 2017 - 2018
- Survey Design: NHANES uses a complex, multistage probability sampling design
- Number of observations: To-be-determined.
- Number of features: To-be-determined.
- Primary analytical approach: Unsupervised learning.
- Prediction target: None for the primary unsupervised analysis.
- Population of interest: NHANES participants meeting the predefined anaemia criteria.
- Analytical classification variable: Anaemia status derived from haemoglobin thresholds.

### 1.4 Feature Description
The nature of the information being investigated involves:
- Biological information such as haematological measurements, iron status, iron metabolism, inflammatory biomarkers, folate, renal, liver, and thyroid function, etc.
-  Contextual variables such as demographic and socioeconomic information from survey data.
A detailed description is made available in the data dictionary.

### 1.5 Scope and Analytical Approach
- The analysis focuses on NHANES participants with anaemia. 
- Multiple biological factors associated with anaemia will be assessed simultaneously.
- Mechanism-related phenotypes of anaemia will be characterised using established biochemical and haematological indicators.
- Methods used to investigate patterns and relationships include exploratory data analysis and unsupervised learning.
- Any findings will be interpreted as associations and patterns consistent with plausible biological mechanisms rather than proof of direct causation or clinical diagnosis.

## 2.0 Dataset Selection Rationale

### 2.1 NHANES Rationale
Data from the NHANES was selected for its public availability, as well as the nature in which information was collected: The NHANES combines three major forms of data collection; Interviews/survey data (demographic, socioeconomic, health and dietary information), physical examination via Mobile Examination Centers (MECs) (measured physical and physiological characteristics) and Laboratory Investigations (biochemical and haematological measurements) providing much more comprehensive information on the physical, biological, and socioeconomic status of each participant.
In addition, it incorporates a complex survey design which consists of probability sampling, multistage sampling, oversampling and corresponding survey weighting, enabling researchers to produce estimates that represent the US civilian, non-institutionalised population.

### 2.2 Survey Cycle Rationale
The 2017-2018 cycle provides a completed pre-pandemic survey cycle, containing a broad range of relevant haematological and biochemical measurements. 

#### 2.2.1 Limitations
The use of a single cycle may not provide sufficient number of observations for subgroup analyses, especially after restricting the population to participants with anaemia, incomplete availability of measurements, laboratory subsampling etc.

Additional survey cycles may be included as the project progresses (variable compatibility and combined-cycle weighting procedures should be applied during this process).

### 2.3 Dataset Selection
NHANES organizes information into five components: Dietary, Demographics, Examination, Laboratory and Questionnaire, with each component containing separate dataset files for various surveys and measurements.
The files are released in survey cycles with alphabetical suffixes identifying each cycle ('J' represents the 2017-2018 cycle). Files from the same cycle can be linked via the participant identifier variable; SEQN. However, laboratory measurements may be restricted to eligible participants or specific subsamples resulting in differing participant counts across files.

Files selected for the analysis include:

- DEMO_J - Demographic information and survey design variables
- CBC_J	- Complete blood count and haematological measurements
- FETIB_J - Iron status measurements (Iron, total iron-bing capacity and other related iron measurements)
- FERTIN_J - Serum ferritin measurements
- TFR_J	- Soluble transferrin receptor measurements (sTfR)
- HSCRP_J - High-sensitivity C-reactive protein
- BIOPRO_J - Biochemical measurements, including renal and liver-related markers
- FOLATE_J - Red blood cell folate measurements


 

## 3.0 Loading and Initial Inspection

### 3.1 Importing Libraries

In [1]:
import pandas as pd
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re

### 3.2 Dataset Assessment
Loading the datasets and creating a summary file and variable metadata table for each data file to assess structural integrity and data quality.

In [2]:
folder = Path(r"..\data")

  
for file in folder.rglob("*.xpt"):
        
        file_df = pd.read_sas(file)
        
        file_name = file.stem
    
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/{file_name}.htm"

        response = requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, "html.parser")

        page_header = soup.find("div", id = "PageHeader")
        # print(page_header)

        file_year_header = page_header.find("h2").get_text()
        file_info = {
            "data" : 'info',
            "file_name" : file.stem,
            "file_year" : re.search(r"\d{4}-\d{4}", file_year_header).group(),
            # "source_url" : url,
            "rows_no" : len(file_df),
            "columns_no" : len(file_df.columns),
            "unique_seqn" : file_df["SEQN"].is_unique,
            "number_of_duplicated_seqn" : file_df["SEQN"].duplicated().sum(),
            "number_of_duplicated_rows" : file_df.duplicated().sum(),
            "number_of_duplicated_column_names" : file_df.columns.duplicated().sum(),
                    }
        # print(columns)
        # print(file_year)

        file_table = pd.DataFrame(file_info, index = [0])
        display(file_table.set_index(file_table.columns[0]).T)


        sub_table_info = []
        for column in file_df.columns:
            sub_file_info = {}
            sub_file_info["code_name"] = column

            
            listed_item_text = soup.find("a", href = f"#{column}").get_text().strip()
            # print(listed_item_text)
            # print(repr(listed_item_text))
            listed_item = re.search(r"^(.+?)\s", listed_item_text).group(1)
            # print(listed_item)
            
            actual_name = re.search(r"\w+\s-\s(.+?)(?:\s\([^)]*\))?$", listed_item_text).group(1)

            sub_file_info["description"] = actual_name

            units = re.search(r"^\w+\s-\s(?:.+?)\s(\([^)]*\))$", listed_item_text)
            if units == None:
                units = "none"
            else:
                units = units.group(1)

            # if file_name in ('CBC_J','FETIB_J'): # issue w/ mean cell hgb conc. and transferrin receptor respectively. solved --> .strip()
            #     print(repr(listed_item_text))
            #     print(actual_name)
            #     print(units)

            # print(units)
            sub_file_info["units"] = units
            sub_file_info["obj_type"] = file_df[column].dtypes
            sub_file_info["number_of_missingness"] = file_df[column].isna().sum()
            sub_file_info["percentage_missingness (%)"] = round((file_df[column].isna().mean())*100, 2)

            # if column == 'SEQN':
            #     sub_file_info["max_value"] = 'none'
            #     sub_file_info["minimum"] = 'none'

            # else:
            #     sub_file_info["max_value"] = file_df[column].max()
            #     sub_file_info["minimum"] = file_df[column].min()
                # sub_file_info["desc"] = 


            sub_table_info.append(sub_file_info)
        sub_file_table = pd.DataFrame(sub_table_info)
        display(sub_file_table)

data,info
file_name,BIOPRO_J
file_year,2017-2018
rows_no,6401
columns_no,41
unique_seqn,True
number_of_duplicated_seqn,0
number_of_duplicated_rows,0
number_of_duplicated_column_names,0


,code_name,description,units,obj_type,number_of_missingness,percentage_missingness (%)
0,SEQN,Respondent Sequence Number,none,float64,0,0.00
1,LBXSATSI,Alanine Aminotransferase (ALT),(U/L),float64,499,7.80
2,LBDSATLC,ALT Comment Code,none,float64,499,7.80
3,LBXSAL,"Albumin, refrigerated serum",(g/dL),float64,496,7.75
4,LBDSALSI,"Albumin, refrigerated serum",(g/L),float64,496,7.75
5,LBXSAPSI,Alkaline Phosphatase (ALP),(IU/L),float64,498,7.78
6,LBXSASSI,Aspartate Aminotransferase (AST),(U/L),float64,519,8.11
7,LBXSC3SI,Bicarbonate,(mmol/L),float64,500,7.81
8,LBXSBU,Blood Urea Nitrogen,(mg/dL),float64,500,7.81
9,LBDSBUSI,Blood Urea Nitrogen,(mmol/L),float64,500,7.81


data,info
file_name,CBC_J
file_year,2017-2018
rows_no,8366
columns_no,22
unique_seqn,True
number_of_duplicated_seqn,0
number_of_duplicated_rows,0
number_of_duplicated_column_names,0


,code_name,description,units,obj_type,number_of_missingness,percentage_missingness (%)
0,SEQN,Respondent sequence number,none,float64,0,0.00
1,LBXWBCSI,White blood cell count,(1000 cells/uL),float64,838,10.02
2,LBXLYPCT,Lymphocyte percent,(%),float64,843,10.08
3,LBXMOPCT,Monocyte percent,(%),float64,843,10.08
4,LBXNEPCT,Segmented neutrophils percent,(%),float64,843,10.08
5,LBXEOPCT,Eosinophils percent,(%),float64,843,10.08
6,LBXBAPCT,Basophils percent,(%),float64,843,10.08
7,LBDLYMNO,Lymphocyte number,(1000 cells/uL),float64,843,10.08
8,LBDMONO,Monocyte number,(1000 cells/uL),float64,843,10.08
9,LBDNENO,Segmented neutrophils num,(1000 cell/uL),float64,843,10.08


data,info
file_name,DEMO_J
file_year,2017-2018
rows_no,9254
columns_no,46
unique_seqn,True
number_of_duplicated_seqn,0
number_of_duplicated_rows,0
number_of_duplicated_column_names,0


,code_name,description,units,obj_type,number_of_missingness,percentage_missingness (%)
0,SEQN,Respondent sequence number,none,float64,0,0.00
1,SDDSRVYR,Data release cycle,none,float64,0,0.00
2,RIDSTATR,Interview/Examination status,none,float64,0,0.00
3,RIAGENDR,Gender,none,float64,0,0.00
4,RIDAGEYR,Age in years at screening,none,float64,0,0.00
5,RIDAGEMN,Age in months at screening - 0 to 24 mos,none,float64,8657,93.55
6,RIDRETH1,Race/Hispanic origin,none,float64,0,0.00
7,RIDRETH3,Race/Hispanic origin w/ NH Asian,none,float64,0,0.00
8,RIDEXMON,Six month time period,none,float64,550,5.94
9,RIDEXAGM,Age in months at exam - 0 to 19 years,none,float64,5821,62.90


... (6 outputs left out)

data,info
file_name,HSCRP_J
file_year,2017-2018
rows_no,8366
columns_no,3
unique_seqn,True
number_of_duplicated_seqn,0
number_of_duplicated_rows,0
number_of_duplicated_column_names,0


,code_name,description,units,obj_type,number_of_missingness,percentage_missingness (%)
0,SEQN,Respondent Sequence Number,none,float64,0,0.00
1,LBXHSCRP,HS C-Reactive Protein,(mg/L),float64,1116,13.34
2,LBDHRPLC,HS C-Reactive Protein Comment Code,none,float64,1116,13.34


data,info
file_name,TFR_J
file_year,2017-2018
rows_no,7332
columns_no,3
unique_seqn,True
number_of_duplicated_seqn,0
number_of_duplicated_rows,0
number_of_duplicated_column_names,0


,code_name,description,units,obj_type,number_of_missingness,percentage_missingness (%)
0,SEQN,Respondent sequence number,none,float64,0,0.00
1,LBXTFR,Transferrin receptor,(mg/L),float64,765,10.43
2,LBDTFRSI,Transferrin receptor,(nmol/L),float64,765,10.43


1. No duplicate rows, column names or SEQN values were identified in the dataset.
2. Missingness in SEQN values was not observed.
3. The number of rows are inconsistent across datasets, suggestive of sampling on a randomized or eligibility basis, procedural incompletion, among others.
4. With the exception of `DEMO_J.xpt`, all other files displayed minimal variation in number and percentage of missingness per non-SEQN variable, ranging from 7.48 to 13.34%. Further investigation may be needed to determine missingness structure.
5. `DEMO_J.xpt` displayed large variation in missingness per variable. Further investigation will be performed after variable selection.
6. Data files such as `DEMO_J.xpt` displayed variables stored as float values, rather than the expected datatype heterogeneity. Further investigation is needed to determine whether this reflects data representation or the true nature of the variables.
7. Multiple variables within `DEMO_J.xpt`, such as RIDAGEMN display missingness percentages as large as 93%
7. SEQN values appear to be in float format.

### 3.3 Previewing the Datasets

In [3]:
folder = Path(r"..\data")

for file in folder.glob("*.xpt"):
        file_df = pd.read_sas(file)
        print(file.stem)
        display(file_df.head())
        # display(file_df.tail())


BIOPRO_J


,SEQN,LBXSATSI,LBDSATLC,LBXSAL,LBDSALSI,LBXSAPSI,LBXSASSI,LBXSC3SI,LBXSBU,LBDSBUSI,...,LBXSCA,LBDSCASI,LBXSCH,LBDSCHSI,LBXSTP,LBDSTPSI,LBXSTR,LBDSTRSI,LBXSUA,LBDSUASI
0,93705.0,16.0,5.397605e-79,4.4,44.0,74.0,20.0,31.0,11.0,3.93,...,9.2,2.300,157.0,4.060,7.3,73.0,95.0,1.073,5.8,345.0
1,93706.0,10.0,5.397605e-79,4.4,44.0,79.0,14.0,28.0,12.0,4.28,...,9.6,2.400,149.0,3.853,7.1,71.0,92.0,1.039,8.0,475.8
2,93707.0,13.0,5.397605e-79,5.2,52.0,238.0,24.0,22.0,17.0,6.07,...,10.1,2.525,199.0,5.146,8.0,80.0,110.0,1.242,5.5,327.1
3,93708.0,19.0,5.397605e-79,3.9,39.0,66.0,21.0,27.0,16.0,5.71,...,9.5,2.375,210.0,5.431,7.1,71.0,72.0,0.813,4.5,267.7
4,93709.0,15.0,5.397605e-79,3.7,37.0,86.0,17.0,24.0,20.0,7.14,...,9.9,2.475,180.0,4.655,7.0,70.0,132.0,1.490,6.2,368.8


CBC_J


,SEQN,LBXWBCSI,LBXLYPCT,LBXMOPCT,LBXNEPCT,LBXEOPCT,LBXBAPCT,LBDLYMNO,LBDMONO,LBDNENO,...,LBXRBCSI,LBXHGB,LBXHCT,LBXMCVSI,LBXMCHSI,LBXMC,LBXRDW,LBXPLTSI,LBXMPSI,LBXNRBC
0,93703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,7.4,47.8,8.0,42.6,1.0,0.7,3.5,0.6,3.2,...,4.25,13.1,37.0,87.0,30.8,35.4,12.8,239.0,8.6,1.000000e-01
2,93705.0,8.6,40.0,7.4,48.8,2.9,1.0,3.4,0.6,4.2,...,5.48,11.9,36.7,67.0,21.7,32.4,15.6,309.0,7.9,5.397605e-79
3,93706.0,6.1,24.6,9.1,61.4,4.3,0.8,1.5,0.6,3.7,...,5.24,16.3,47.0,89.7,31.1,34.7,12.2,233.0,6.6,5.397605e-79
4,93707.0,11.2,37.1,6.2,54.7,1.6,0.5,4.2,0.7,6.1,...,5.02,14.5,42.1,83.9,28.9,34.4,13.6,348.0,8.5,2.000000e-01


DEMO_J


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,93703.0,10.0,2.0,2.0,2.0,NaN,5.0,6.0,2.0,27.0,...,3.0,1.0,3.0,9246.491865,8539.731348,2.0,145.0,15.0,15.0,5.00
1,93704.0,10.0,2.0,1.0,2.0,NaN,3.0,3.0,1.0,33.0,...,3.0,1.0,2.0,37338.768343,42566.614750,1.0,143.0,15.0,15.0,5.00
2,93705.0,10.0,2.0,2.0,66.0,NaN,4.0,4.0,2.0,NaN,...,1.0,2.0,NaN,8614.571172,8338.419786,2.0,145.0,3.0,3.0,0.82
3,93706.0,10.0,2.0,1.0,18.0,NaN,5.0,6.0,2.0,222.0,...,3.0,1.0,2.0,8548.632619,8723.439814,2.0,134.0,NaN,NaN,NaN
4,93707.0,10.0,2.0,1.0,13.0,NaN,5.0,7.0,2.0,158.0,...,2.0,1.0,3.0,6769.344567,7064.609730,1.0,138.0,10.0,10.0,1.88


... (6 outputs left out)

HSCRP_J


,SEQN,LBXHSCRP,LBDHRPLC
0,93703.0,NaN,NaN
1,93704.0,0.29,5.397605e-79
2,93705.0,2.72,5.397605e-79
3,93706.0,0.74,5.397605e-79
4,93707.0,0.32,5.397605e-79


TFR_J


,SEQN,LBXTFR,LBDTFRSI
0,93703.0,NaN,NaN
1,93704.0,2.59,30.6
2,93705.0,3.37,39.8
3,93706.0,2.88,34.0
4,93707.0,2.88,34.0


1. Certain variables such as LBDSATLC (Alanine Amino Transferase - ALT Comment Code), LBXNRBC (Nucleated red blood cells), LBDUIBLC (UIBC, Serum Comment Code), etc., repeatedly display unusually small value of 5.397605e-79, contrary to value-meanings in codebook documentation. Further investigation is needed to determine if it is a special value placeholder for "Comment code" variables, an issue wth the import process etc.
2. Rows of missing values, particularly for SEQN 93703 are observed in multiple datasets, likely caused by factors such as sampling eligibility for certain tests.
3. `FERTIN_J.xpt` displayed the same values for its non-SEQN variables. Information from the Metadata table above confirms numeric equivalence as their unit conversion factors cancel out.


In [4]:
folder = Path(r"..\data")

for file in folder.glob("*.xpt"):
        file_df = pd.read_sas(file)
        print(f"{file.stem}\n")

        for column in file_df.columns:         
            if column.endswith("LC"):
                    print(f"{file_df[column].unique()}\n{file_df[column].value_counts(dropna = False)}\n")

BIOPRO_J

[5.39760535e-79            nan 1.00000000e+00]
LBDSATLC
5.397605e-79    5901
NaN              499
1.000000e+00       1
Name: count, dtype: int64

[5.39760535e-79            nan 1.00000000e+00]
LBDSGTLC
5.397605e-79    5901
NaN              499
1.000000e+00       1
Name: count, dtype: int64

[5.39760535e-79            nan 1.00000000e+00]
LBDSTBLC
5.397605e-79    5902
NaN              498
1.000000e+00       1
Name: count, dtype: int64

CBC_J

DEMO_J

FERTIN_J

FETIB_J

[5.39760535e-79            nan 1.00000000e+00]
LBDUIBLC
5.397605e-79    5893
NaN              502
1.000000e+00       6
Name: count, dtype: int64

FOLATE_J

HSCRP_J

[           nan 5.39760535e-79 1.00000000e+00]
LBDHRPLC
5.397605e-79    7175
NaN             1116
1.000000e+00      75
Name: count, dtype: int64

TFR_J

